In [1]:
import os
import random
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from scipy.optimize import minimize


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
print(f"Исходные данные загружены. Train: {train.shape}, Test: {test.shape}")


Исходные данные загружены. Train: (751, 214), Test: (250, 211)


In [3]:
target_cols = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in train.columns if col not in ["index"] + target_cols]


In [4]:
medians = train.groupby(feature_cols)[target_cols].transform('median')
train[target_cols] = medians
train_cleaned = train.drop_duplicates(subset=feature_cols, keep='first').reset_index(drop=True)
print(f"После объединения дубликатов молекул по медиане: {train_cleaned.shape}")


После объединения дубликатов молекул по медиане: (630, 214)


In [5]:
train_cleaned = train_cleaned.dropna(subset=target_cols).reset_index(drop=True)
print(f"После удаления строк с пустыми таргетами (NaN): {train_cleaned.shape}")


После удаления строк с пустыми таргетами (NaN): (628, 214)


In [6]:
selector = VarianceThreshold(threshold=0.0)
selector.fit(train_cleaned[feature_cols])
constant_features = [col for col, keep in zip(feature_cols, selector.get_support()) if not keep]
feature_cols = [col for col in feature_cols if col not in constant_features]
print(f"Удалено константных признаков: {len(constant_features)}. Осталось признаков: {len(feature_cols)}")


Удалено константных признаков: 18. Осталось признаков: 192


In [7]:
corr_matrix = train_cleaned[feature_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_features = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]
feature_cols = [col for col in feature_cols if col not in high_corr_features]
print(f"Удалено сильно коррелирующих признаков (>0.95): {len(high_corr_features)}. Итого признаков: {len(feature_cols)}")


Удалено сильно коррелирующих признаков (>0.95): 34. Итого признаков: 158


In [8]:
q25 = train_cleaned['SI'].quantile(0.25)
q75 = train_cleaned['SI'].quantile(0.75)
iqr = q75 - q25
upper_boundary = q75 + 3.0 * iqr
train_final = train_cleaned[train_cleaned['SI'] <= upper_boundary].reset_index(drop=True)
print(f"Граница выбросов по IQR для SI: {upper_boundary:.2f}. Удалено выбросов: {train_cleaned.shape[0] - train_final.shape[0]}")


Граница выбросов по IQR для SI: 48.59. Удалено выбросов: 49


In [9]:
train_final['fold'] = -1
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(train_final)):
    train_final.loc[val_idx, 'fold'] = fold_idx

print(f"\n Очищенный датасет")
print(f"Размерность train_final: {train_final.shape}")
print(f"Количество признаков в feature_cols: {len(feature_cols)}")
print(f"Распределение строк по 5 фолдам:\n{train_final['fold'].value_counts().to_string()}")


 Очищенный датасет
Размерность train_final: (579, 215)
Количество признаков в feature_cols: 158
Распределение строк по 5 фолдам:
fold
1    116
0    116
2    116
3    116
4    115


C:\Users\Professional\AppData\Local\Temp\ipykernel_15684\3667698995.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_final['fold'] = -1


In [10]:
train_final = train_final.copy()


In [11]:
models_ic50 = []
models_cc50 = []
models_si = []

train_final['oof_IC50'] = 0.0
train_final['oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
              eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM']),
              early_stopping_rounds=100, verbose=False)

    train_final.loc[val_idx, 'oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
              eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM']),
              early_stopping_rounds=100, verbose=False)

    train_final.loc[val_idx, 'oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    models_cc50.append(model)

feature_cols_si = feature_cols + ['oof_IC50', 'oof_CC50']
for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols_si], train_final.loc[train_idx, 'SI'],
              eval_set=(train_final.loc[val_idx, feature_cols_si], train_final.loc[val_idx, 'SI']),
              early_stopping_rounds=100, verbose=False)
    models_si.append(model)

test_preds_ic50 = np.zeros(len(test))
test_preds_cc50 = np.zeros(len(test))
test_preds_si = np.zeros(len(test))

for model in models_ic50:
    test_preds_ic50 += model.predict(test[feature_cols]) / 5

for model in models_cc50:
    test_preds_cc50 += model.predict(test[feature_cols]) / 5

test_meta = test.copy()
test_meta['oof_IC50'] = test_preds_ic50
test_meta['oof_CC50'] = test_preds_cc50

for model in models_si:
    test_preds_si += model.predict(test_meta[feature_cols_si]) / 5


submission = pd.DataFrame({
    'index': test['index'],
    'IC50': test_preds_ic50,
    'CC50': test_preds_cc50,
    'SI': test_preds_si
})

submission.to_csv('submission.csv', index=False)
print("Файл submission.csv создан. Формат:")
print(submission.head())
print(f"\nРазмерность файла: {submission.shape}")

Файл submission.csv создан. Формат:
   index        IC50        CC50        SI
0      0  173.109074  360.244931  8.202102
1      1  235.091709  379.337866  5.609795
2      2  158.658089  305.884217  8.568503
3      3  281.977935  396.177885  6.447149
4      4  207.574996  357.836030  4.830452

Размерность файла: (250, 4)


In [12]:
xgb_models_ic50 = []
xgb_models_cc50 = []
xgb_models_si = []

train_final['xgb_oof_IC50'] = 0.0
train_final['xgb_oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])],
        verbose=False
    )

    train_final.loc[val_idx, 'xgb_oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    xgb_models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])],
        verbose=False
    )

    train_final.loc[val_idx, 'xgb_oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    xgb_models_cc50.append(model)

feature_cols_xgb_si = feature_cols + ['xgb_oof_IC50', 'xgb_oof_CC50']
xgb_oof_predictions_si = np.zeros(len(train_final))

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols_xgb_si], train_final.loc[train_idx, 'SI'],
        eval_set=[(train_final.loc[val_idx, feature_cols_xgb_si], train_final.loc[val_idx, 'SI'])],
        verbose=False
    )
    xgb_oof_predictions_si[val_idx] = model.predict(train_final.loc[val_idx, feature_cols_xgb_si])
    xgb_models_si.append(model)

xgb_test_ic50 = np.zeros(len(test))
xgb_test_cc50 = np.zeros(len(test))
xgb_test_si = np.zeros(len(test))

for model in xgb_models_ic50:
    xgb_test_ic50 += model.predict(test[feature_cols]) / 5

for model in xgb_models_cc50:
    xgb_test_cc50 += model.predict(test[feature_cols]) / 5

test_meta_xgb = test.copy()
test_meta_xgb['xgb_oof_IC50'] = xgb_test_ic50
test_meta_xgb['xgb_oof_CC50'] = xgb_test_cc50

for model in xgb_models_si:
    xgb_test_si += model.predict(test_meta_xgb[feature_cols_xgb_si]) / 5

print("Обучение XGBoost завершено")
print(f"Локальный OOF RMSE для IC50 (XGB): {root_mean_squared_error(train_final['IC50, mM'], train_final['xgb_oof_IC50']):.4f}")
print(f"Локальный OOF RMSE для CC50 (XGB): {root_mean_squared_error(train_final['CC50, mM'], train_final['xgb_oof_CC50']):.4f}")
print(f"Локальный OOF RMSE для SI (XGB): {root_mean_squared_error(train_final['SI'], xgb_oof_predictions_si):.4f}")


Обучение XGBoost завершено
Локальный OOF RMSE для IC50 (XGB): 330.7210
Локальный OOF RMSE для CC50 (XGB): 451.3674
Локальный OOF RMSE для SI (XGB): 9.5953


In [13]:
blended_ic50 = (test_preds_ic50 + xgb_test_ic50) / 2
blended_cc50 = (test_preds_cc50 + xgb_test_cc50) / 2
blended_si = (test_preds_si + xgb_test_si) / 2
submission_blended = pd.DataFrame({
    'index': test['index'],
    'IC50': blended_ic50,
    'CC50': blended_cc50,
    'SI': blended_si
})

submission_blended.to_csv('submission_blended.csv', index=False)

print("Файл submission_blended.csv создан")
print(submission_blended.head())

Файл submission_blended.csv создан
   index        IC50        CC50        SI
0      0  183.376287  407.301821  8.574248
1      1  237.705301  390.244605  5.849062
2      2  153.684235  346.939434  8.095828
3      3  303.942306  438.748698  6.133202
4      4  198.019498  353.846655  5.372371


In [14]:
lgb_models_ic50 = []
lgb_models_cc50 = []
lgb_models_si = []

train_final['lgb_oof_IC50'] = 0.0
train_final['lgb_oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    train_final.loc[val_idx, 'lgb_oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    lgb_models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    train_final.loc[val_idx, 'lgb_oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    lgb_models_cc50.append(model)

feature_cols_lgb_si = feature_cols + ['lgb_oof_IC50', 'lgb_oof_CC50']
lgb_oof_predictions_si = np.zeros(len(train_final))

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        train_final.loc[train_idx, feature_cols_lgb_si], train_final.loc[train_idx, 'SI'],
        eval_set=[(train_final.loc[val_idx, feature_cols_lgb_si], train_final.loc[val_idx, 'SI'])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    lgb_oof_predictions_si[val_idx] = model.predict(train_final.loc[val_idx, feature_cols_lgb_si])
    lgb_models_si.append(model)

lgb_test_ic50 = np.zeros(len(test))
lgb_test_cc50 = np.zeros(len(test))
lgb_test_si = np.zeros(len(test))

for model in lgb_models_ic50:
    lgb_test_ic50 += model.predict(test[feature_cols]) / 5

for model in lgb_models_cc50:
    lgb_test_cc50 += model.predict(test[feature_cols]) / 5

test_meta_lgb = test.copy()
test_meta_lgb['lgb_oof_IC50'] = lgb_test_ic50
test_meta_lgb['lgb_oof_CC50'] = lgb_test_cc50

for model in lgb_models_si:
    lgb_test_si += model.predict(test_meta_lgb[feature_cols_lgb_si]) / 5

print("Обучение LightGBM завершено")
print(f"Локальный OOF RMSE для IC50 (LGB): {root_mean_squared_error(train_final['IC50, mM'], train_final['lgb_oof_IC50']):.4f}")
print(f"Локальный OOF RMSE для CC50 (LGB): {root_mean_squared_error(train_final['CC50, mM'], train_final['lgb_oof_CC50']):.4f}")
print(f"Локальный OOF RMSE для SI (LGB): {root_mean_squared_error(train_final['SI'], lgb_oof_predictions_si):.4f}")


Обучение LightGBM завершено
Локальный OOF RMSE для IC50 (LGB): 328.7775
Локальный OOF RMSE для CC50 (LGB): 456.3897
Локальный OOF RMSE для SI (LGB): 9.5069


In [15]:
triple_blended_ic50 = (test_preds_ic50 + xgb_test_ic50 + lgb_test_ic50) / 3
triple_blended_cc50 = (test_preds_cc50 + xgb_test_cc50 + lgb_test_cc50) / 3
triple_blended_si = (test_preds_si + xgb_test_si + lgb_test_si) / 3

submission_triple = pd.DataFrame({
    'index': test['index'],
    'IC50': triple_blended_ic50,
    'CC50': triple_blended_cc50,
    'SI': triple_blended_si
})

submission_triple.to_csv('submission_triple_blend.csv', index=False)

print("Файл submission_triple_blend.csv создан")
print(submission_triple.head())


Файл submission_triple_blend.csv создан
   index        IC50        CC50        SI
0      0  191.264482  397.777092  8.415824
1      1  241.069216  398.769699  5.985821
2      2  157.073158  412.768885  8.002162
3      3  306.050083  410.405055  5.950809
4      4  209.663371  326.389420  5.233820


In [ ]:

best_weights = {}

for target_name, cb_col, xgb_col, lgb_col in [
    ('IC50', 'oof_IC50', 'xgb_oof_IC50', 'lgb_oof_IC50'),
    ('CC50', 'oof_CC50', 'xgb_oof_CC50', 'lgb_oof_CC50')
]:
    real_target_col = target_name + ', mM'
    y_true = train_final[real_target_col]

    cb_oof = train_final[cb_col]
    xgb_oof = train_final[xgb_col]
    lgb_oof = train_final[lgb_col]

    def rmse_func(weights):
        w1, w2, w3 = weights
        pred = w1 * cb_oof + w2 * xgb_oof + w3 * lgb_oof
        return root_mean_squared_error(y_true, pred)

    constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
    bounds = [(0, 1), (0, 1), (0, 1)]
    initial_weights = [1/3, 1/3, 1/3]

    res = minimize(rmse_func, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
    best_weights[target_name] = np.round(res.x, 3)

w_ic50_cb, w_ic50_xgb, w_ic50_lgb = best_weights['IC50']
w_cc50_cb, w_cc50_xgb, w_cc50_lgb = best_weights['CC50']

print(f"Идеальные веса для IC50 (CB/XGB/LGB): {w_ic50_cb} / {w_ic50_xgb} / {w_ic50_lgb}")
print(f"Идеальные веса для CC50 (CB/XGB/LGB): {w_cc50_cb} / {w_cc50_xgb} / {w_cc50_lgb}")

weighted_test_ic50 = w_ic50_cb * test_preds_ic50 + w_ic50_xgb * xgb_test_ic50 + w_ic50_lgb * lgb_test_ic50
weighted_test_cc50 = w_cc50_cb * test_preds_cc50 + w_cc50_xgb * xgb_test_cc50 + w_cc50_lgb * lgb_test_cc50


print("\n OOF для SI от всех трех моделей")
train_final['cb_oof_SI'] = 0.0
train_final['xgb_oof_SI'] = 0.0
train_final['lgb_oof_SI'] = 0.0

feature_cols_si = feature_cols + ['oof_IC50', 'oof_CC50']
for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index
    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols_si], train_final.loc[train_idx, 'SI'],
              eval_set=(train_final.loc[val_idx, feature_cols_si], train_final.loc[val_idx, 'SI']),
              early_stopping_rounds=100, verbose=False)
    train_final.loc[val_idx, 'cb_oof_SI'] = model.predict(train_final.loc[val_idx, feature_cols_si])

feature_cols_xgb_si = feature_cols + ['xgb_oof_IC50', 'xgb_oof_CC50']
for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index
    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(train_final.loc[train_idx, feature_cols_xgb_si], train_final.loc[train_idx, 'SI'],
              eval_set=[(train_final.loc[val_idx, feature_cols_xgb_si], train_final.loc[val_idx, 'SI'])], verbose=False)
    train_final.loc[val_idx, 'xgb_oof_SI'] = model.predict(train_final.loc[val_idx, feature_cols_xgb_si])

feature_cols_lgb_si = feature_cols + ['lgb_oof_IC50', 'lgb_oof_CC50']
for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index
    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(train_final.loc[train_idx, feature_cols_lgb_si], train_final.loc[train_idx, 'SI'],
              eval_set=[(train_final.loc[val_idx, feature_cols_lgb_si], train_final.loc[val_idx, 'SI'])],
              callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    train_final.loc[val_idx, 'lgb_oof_SI'] = model.predict(train_final.loc[val_idx, feature_cols_lgb_si])

y_true_si = train_final['SI']
cb_si, xgb_si, lgb_si = train_final['cb_oof_SI'], train_final['xgb_oof_SI'], train_final['lgb_oof_SI']

def rmse_si_func(weights):
    w1, w2, w3 = weights
    pred = w1 * cb_si + w2 * xgb_si + w3 * lgb_si
    return root_mean_squared_error(y_true_si, pred)

res_si = minimize(rmse_si_func, [1/3, 1/3, 1/3], method='SLSQP', bounds=[(0, 1), (0, 1), (0, 1)], constraints=({'type': 'eq', 'fun': lambda w: 1 - sum(w)}))
w_si_cb, w_si_xgb, w_si_lgb = np.round(res_si.x, 3)
print(f"Идеальные веса для SI (CB/XGB/LGB): {w_si_cb} / {w_si_xgb} / {w_si_lgb}")

weighted_test_si = w_si_cb * test_preds_si + w_si_xgb * xgb_test_si + w_si_lgb * lgb_test_si

submission_weighted = pd.DataFrame({
    'index': test['index'],
    'IC50': weighted_test_ic50,
    'CC50': weighted_test_cc50,
    'SI': weighted_test_si
})

submission_weighted.to_csv('submission_weighted_blend.csv', index=False)
print("\n submission_weighted_blend.csv создан")
print(submission_weighted.head())


Идеальные веса для IC50 (CB/XGB/LGB): 0.269 / 0.321 / 0.41
Идеальные веса для CC50 (CB/XGB/LGB): 1.0 / 0.0 / 0.0

 OOF для SI от всех трех моделей
Идеальные веса для SI (CB/XGB/LGB): 0.569 / 0.076 / 0.355

 submission_weighted_blend.csv создан
   index        IC50        CC50        SI
0      0  193.612662  360.244931  8.222059
1      1  241.978823  379.337866  5.876752
2      2  157.593971  305.884217  8.229102
3      3  307.677019  396.177885  6.093729
4      4  211.844575  357.836030  4.957649
